# AF2 spectral — winner confirmation
Attach the global-decision ZIP and private core dataset. Set only `SEED` to 123 or 2026; run this notebook once per seed/account.

In [ ]:
import json, os, shutil, subprocess, sys, time
from pathlib import Path
SEED=123  # change only to 2026 for the second paired confirmation
assert SEED in (123,2026)
WORK=Path('/kaggle/working'); os.chdir(WORK); REPO=WORK/'coffee-bean-detection'; INPUT=Path('/kaggle/input'); OUT=WORK/'af2-spectral-factorization-v1'
if REPO.exists(): shutil.rmtree(REPO)
for _ in range(3):
 if subprocess.run(['git','clone','--depth','1','--branch','agent/af2-spectral-factorization','https://github.com/ediprin/coffee-bean-detection.git',str(REPO)]).returncode==0: break
 if REPO.exists(): shutil.rmtree(REPO)
else: raise RuntimeError('git clone gagal tiga kali')
subprocess.run([sys.executable,'-m','pip','install','-q','ultralytics==8.4.96','-e',str(REPO)],check=True); os.chdir(REPO)
sys.path.insert(0,str(REPO/'src'))
for module_name in list(sys.modules):
    if module_name == 'coffee_detector' or module_name.startswith('coffee_detector.'): sys.modules.pop(module_name,None)
from coffee_detector.experiments.prepare_af2_spectral_kaggle import prepare_af2_spectral_kaggle_input
DATA,A,_=prepare_af2_spectral_kaggle_input(INPUT,WORK); OUT.mkdir(exist_ok=True); (OUT/'val_reports').mkdir(exist_ok=True)
matches=sorted(INPUT.rglob('global_seed42_decision.json')); assert len(matches)==1,matches; GLOBAL=matches[0]; decision=json.loads(GLOBAL.read_text()); assert decision['decision']=='PASS' and decision['test_opened'] is False; ARM=decision['winner']
D0=A[f'D0_seed{SEED}_best.pt']; LOG=OUT/f'confirmation_{ARM}_seed{SEED}.log'; RESULT=OUT/'val_reports'/f'confirmation_{ARM}_seed{SEED}_result.json'
CMD=[sys.executable,'-u','-m','coffee_detector.experiments.run_faruq_v3_af2_spectral_confirmation','arm','--data-root',str(DATA),'--grouped-summary',str(DATA/'faruq_grouped_summary.json'),'--d0-checkpoint',str(D0),'--global-decision',str(GLOBAL),'--output-root',str(OUT),'--seed',str(SEED),'--device','0','--authorize-training']
if not RESULT.is_file():
 with LOG.open('a') as s: p=subprocess.Popen(CMD,cwd=REPO,stdout=s,stderr=subprocess.STDOUT)
 seen=-1
 while p.poll() is None:
  q=OUT/'confirmation'/ARM/f'{ARM}_seed{SEED}'/'results.csv'; n=max(0,len(q.read_text().splitlines())-1) if q.is_file() else 0
  if n!=seen: print(f'{ARM}/seed{SEED}: {n}/50 epoch | log={LOG}',flush=True); seen=n
  time.sleep(120)
 if p.returncode: print('\n'.join(LOG.read_text(errors='replace').splitlines()[-120:])); raise RuntimeError('confirmation gagal')
assert RESULT.is_file(); print(RESULT.read_text()); print('DOWNLOAD SEBELUM STOP SESSION:',shutil.make_archive(f'/kaggle/working/{ARM}_seed{SEED}_confirmation','zip',OUT))